## DeepEval evalutation

Uses manually-checked and approved goldens only

Install requirements

In [0]:
%pip install -r requirements-notebook.txt
dbutils.library.restartPython()

Imports, constants for baseline eval

In [0]:
import pandas as pd
import time
import random
import mlflow
import json
from langchain_groq import ChatGroq
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from deepeval.models import DeepEvalBaseLLM
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from deepeval.metrics import AnswerRelevancyMetric, FaithfulnessMetric, GEval
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

from tests.custom_groq_model import CustomGroqModel

INDEX_PATH = "faiss_index"
HF_MODEL = "all-MiniLM-L6-v2"
GROQ_MODEL = "openai/gpt-oss-20b"
JUDGE_MODEL = "qwen/qwen3.8-27b"
CHECKPOINT_FILE = "test_sets/evaluation_checkpoint.json"

In [0]:
Log traces for the chatbot model

In [0]:
mlflow.set_experiment("/Shared/fca_cobs_rag_benchmark")
mlflow.langchain.autolog(log_traces=True)

Get Groq API key if necessary

In [0]:
import os
from getpass import getpass
if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API key: ")

Quick glance at the approved goldens

In [0]:
goldens_df = pd.read_csv("test_sets/synthetic_goldens_reviewed.csv")
approved_goldens = goldens_df[goldens_df['approved'] == True][['cid', 'question', 'answer']]

print(approved_goldens.head())
print(f"\n{len(approved_goldens)} approved goldens out of {len(goldens_df)} total")

Set model for the chatbot itself

In [0]:
llm = ChatGroq(model=GROQ_MODEL, temperature=0.0)

Set the prompt (matched to streamlit_app.py prompt) and answer function

In [0]:
from langchain_core.prompts import ChatPromptTemplate

prompt_template = """You are a professional UK Financial Compliance Assistant. 
Answer the user's question using ONLY the provided context from the FCA COBS handbook (Chapters 1-10A).

CRITICAL INSTRUCTIONS:
1. Do NOT start your answer with phrases like "Based on the context," "Based on the provided text," or "According to the documents."
2. Start your answer **directly** with the factual information.
3. If the answer is not in the context, simply state: "I cannot find this information in the provided COBS chapters."
4. Do not hallucinate. Be precise and professional.

Context:
{context}

Question: {question}

Direct answer:"""

rag_prompt = ChatPromptTemplate.from_template(prompt_template)

def rag_answer(question, context_list):
    """single-turn RAG response otherwise matching Streamlit prompt"""
    context_text = chr(10).join(context_list)
    formatted_prompt = rag_prompt.format(context=context_text, question=question)
    response = llm.invoke(formatted_prompt)
    return StrOutputParser().invoke(response)

Load the vector embeddings and initialise the retriever

In [0]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = HuggingFaceEmbeddings(model_name=HF_MODEL)
vector_store = FAISS.load_local(
    INDEX_PATH, 
    embeddings, 
    allow_dangerous_deserialization=True
)

retriever = vector_store.as_retriever(search_kwargs={"k": 6})



Initialise the judge models (using my custom wrapper to play nicely with DeepEval).
Two judges, as Faithfulness metric is more computationaly expensive and typically requires more tokens.

In [0]:
light_judge = CustomGroqModel(
    model=JUDGE_MODEL, 
    seconds_delay=60,  # TPM delay
    temperature=0.0,
    max_tokens = 1024
)

# extra tokens and time for eg Faithfulness metric
heavy_judge = CustomGroqModel(
    model=JUDGE_MODEL, 
    seconds_delay=90,  # extra TPM delay
    temperature=0.0,
    max_tokens=2048 # extra tokens
)

Set up metrics

In [0]:
relevancy_metric = AnswerRelevancyMetric(
    model=light_judge, 
    threshold=0.5, 
    include_reason=True
)

faithfulness_metric = FaithfulnessMetric(
    model=heavy_judge, 
    threshold=0.5, 
    include_reason=True
)

tone_metric = GEval(
    name="Tone",
    criteria="Does the answer maintain a professional compliance tone AND correctly reference or cite the relevant COBS regulation number from the context?",
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    strict_mode=True,
    model=light_judge
)


Checkpoints using JSON for results logged (quota-friendly).  Define run_eval

In [0]:
def load_checkpoint():
    if os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE, "r") as f:
            data = json.load(f)
        return data.get("evaluated_cids", []), data.get("results", [])
    return [], []

def save_checkpoint(evaluated_cids, results):
    with open(CHECKPOINT_FILE, "w") as f:
        json.dump({
            "evaluated_cids": evaluated_cids,
            "results": results
        }, f, indent=2)

def run_eval(csv_path, batch_size=1):
    evaluated_cids, results = load_checkpoint()
    print(f"{'='*60}")
    print(f"Checkpoint: {len(evaluated_cids)} goldens evaluated")
    print(f"{'='*60}")
    
    df = pd.read_csv(csv_path)
    approved = df[df['approved'] == True].copy()
    untested = approved[~approved['cid'].isin(evaluated_cids)]
    
    if len(untested) == 0:
        print("All approved goldens evaluated!")
        return results
    
    batch = untested.head(batch_size)
    print(f"Evaluating {len(batch)} goldens with CustomGroqModel...")
    
    batch_results = []
    
    for idx, row in batch.iterrows():
        cid = row['cid']
        
        print(f"\n[{idx+1}/{len(batch)}] cid={cid}")
        
        # Run pipeline
        retrieved_docs = retriever.invoke(row['question'])
        context = [d.page_content for d in retrieved_docs]
        actual_output = rag_answer(row['question'], context)
        
        case = LLMTestCase(
            input=row['question'],
            actual_output=actual_output,
            expected_output=row['answer'],
            retrieval_context=context
        )
        
        print(f"Input: {case.input[:80]}...")
        print(f"Expected: {case.expected_output[:80]}...")
        print(f"Actual: {case.actual_output[:80]}...")

        # Evaluate metrics
        try:
            print(f"  → Answer relevance...")
            score_relevance = relevancy_metric.measure(case)
            print(f"     Score: {score_relevance:.2f}")
            print(f"     Reason: {relevancy_metric.reason[:80]}...")
        except Exception as e:
            print(f"  Relevance failed: {e}")
            score_relevance = None

        try:
            print(f"  → Faithfulness...")
            score_faithfulness = faithfulness_metric.measure(case)
            print(f"     Score: {score_faithfulness:.2f}")
            print(f"     Reason: {faithfulness_metric.reason[:80]}...")
        except Exception as e:
            print(f"  Faithfulness failed: {e}")
            score_faithfulness = None

        try:
            print(f"  → Tone/Citation (combined GEval)...")
            score_tone = tone_metric.measure(case)
            print(f"     Score: {score_tone:.2f}")
            print(f"     Reason: {tone_metric.reason[:80]}...")
        except Exception as e:
            print(f"  Tone/Citation failed: {e}")
            score_tone = None
        
        result_row = {
            "cid": cid,
            "question": row['question'],
            "answer": case.actual_output,
            "expected_output": row['answer'],
            "answer_relevancy": score_relevance,
            "relevance_reason": relevancy_metric.reason if score_relevance is not None else None,
            "faithfulness": score_faithfulness,
            "faithfulness_reason": faithfulness_metric.reason if score_faithfulness is not None else None,
            "tone_citation": score_tone,
            "tone_citation_reason": tone_metric.reason if score_tone is not None else None,
        }
        batch_results.append(result_row)
        
        # Save checkpoint
        if score_relevance is not None or score_faithfulness is not None or score_tone is not None:
            evaluated_cids.append(cid)
            results.append(result_row) 
            save_checkpoint(evaluated_cids, results)
    
    return results

Run a batched evaluation

In [0]:
results = run_eval("test_sets/synthetic_goldens_reviewed.csv", batch_size=9)

View main evaluation results

In [0]:
with open(CHECKPOINT_FILE, "r") as f:
    checkpoint_data = json.load(f)

checkpoint_results = checkpoint_data.get("results", [])

if not checkpoint_results:
    print("No results found in checkpoint")
else:
    results_df = pd.DataFrame(checkpoint_results)
    
    # summary stats
    avg_relevance = results_df["answer_relevancy"].mean(numeric_only=True)
    avg_faithfulness = results_df["faithfulness"].mean(numeric_only=True)
    avg_tone_citation = results_df["tone_citation"].mean(numeric_only=True)
    
    print(f"\n{'='*60}")
    print(f"EVALUATION SUMMARY ({len(results_df)} goldens)")
    print(f"{'='*60}")
    print(f"  Answer relevancy:   {avg_relevance:.2f}")
    print(f"  Faithfulness:       {avg_faithfulness:.2f}")
    print(f"  Tone and citation:  {avg_tone_citation:.2f}")
    print(f"{'='*60}")

null_metrics = ["answer_relevancy", "faithfulness", "tone_citation"]

for metric in null_metrics:
    null_rows = results_df[results_df[metric].isna()]
    if len(null_rows) > 0:
        null_cids = null_rows["cid"].tolist()
        print(f"\n{metric} nulls for cids: {null_cids}")
        for _, row in null_rows.iterrows():
            print(f"  cid={row['cid']} | question: {row['question'][:80]}...")

MLFlow logging

In [0]:
# initial MLFlow logging for DeepEval evaluation
mlflow.set_experiment("/Shared/fca_cobs_rag_benchmark_eval")

with mlflow.start_run(run_name="golden-evaluation") as run:
    # log params (expand later)
    mlflow.log_param("judge_model", JUDGE_MODEL)
    mlflow.log_param("generator_model", GROQ_MODEL)
    mlflow.log_param("num_goldens_evaluated", len(results_df))
    mlflow.log_param("threshold", 0.5)
    
    # log summary metrics
    mlflow.log_metric("avg_answer_relevancy", round(avg_relevance, 4))
    mlflow.log_metric("avg_faithfulness", round(avg_faithfulness, 4))
    mlflow.log_metric("avg_tone_citation", round(avg_tone_citation, 4))
    mlflow.log_metric("std_answer_relevancy", results_df["answer_relevancy"].std(numeric_only=True))
    mlflow.log_metric("std_faithfulness", results_df["faithfulness"].std(numeric_only=True))
    mlflow.log_metric("std_tone_citation", results_df["tone_citation"].std(numeric_only=True))
    
    # log results
    mlflow.log_table(results_df, "evaluation_results.parquet")
    mlflow.log_artifact(checkpoint_file, artifact_path="results")
    
    # log nulls if any
    # Log null details to MLFlow as a separate artifact
    null_details = []
    for metric in null_metrics:
        null_rows = results_df[results_df[metric].isna()]
        for _, row in null_rows.iterrows():
            null_details.append({
                "cid": row["cid"],
                "question": row["question"],
                "metric_with_null": metric,
                "answer_relevancy": row["answer_relevancy"],
                "faithfulness": row["faithfulness"],
                "tone_citation": row["tone_citation"],
            })

    if null_details:
        null_df = pd.DataFrame(null_details)
        mlflow.log_table(null_df, "null_metrics_details.json")
        print(f"\nLogged {len(null_details)} null entries to MLFlow")
    else:
        print("\nNo nulls found — all metrics scored successfully")

    run_id = run.info.run_id
        
print(f"\nLogged to MLFlow run ID: {run_id}")